# Assignment 5 — Option B: Job Fit Analyzer
## BSAN 6200: Text Mining & Social Media Analytics — Spring 2026

**Student Name:** Shahzeb Ather  
**Date:** 04/24/2026
**Option:** B — Job Fit Analyzer  
**API Path:** Free

---

### Table of Contents
1. [Setup and Imports](#1-setup)
2. [Load Job Descriptions and Resume](#2-loading)
3. [Text Chunking](#3-chunking)
4. [Embedding and Vector Store](#4-embedding)
5. [Analysis Prompts and Chain](#5-analysis)
6. [Zero-shot vs. Few-shot Comparison](#6-comparison)
7. [Evaluation](#7-evaluation)

> **Reminder:** The Streamlit app is a separate file (`streamlit_app.py`). This notebook builds and tests the analysis pipeline.  
> See the Option B Implementation Guide for detailed step requirements.

---
<a id="1-setup"></a>
## 1. Setup and Imports

Install required packages and load your API key from a `.env` file.  
**Do NOT hardcode API keys in this notebook.**

Suggested packages: `langchain`, `langchain-openai` or `langchain-community`, `chromadb` or `faiss-cpu`, `pypdf`, `python-dotenv`, `pandas`, `sentence-transformers` (free path)

In [1]:
#Install packages
!python3 -m pip install streamlit chromadb huggingface-hub python-dotenv sentence-transformers pypdf pandas jupyterlab ipykernel

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/12.5 MB ? eta -:--:--

   ━━━━━━━━━━━━━━━━╸━━━━━━━━━━━━━━━━━━━━━━━ 5.2/12.5 MB 31.8 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.5/12.5 MB 36.4 MB/s  0:00:00


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/10.2 MB ? eta -:--:--

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 49.3 MB/s  0:00:00


...

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╺━ 58/61 [jupyterlab-server]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╺ 60/61 [jupyterlab]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╺ 60/61 [jupyterlab]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61/61 [jupyterlab]



[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: pip3 install --upgrade pip


In [2]:
#Load API keys from .env
import os
import pandas as pd
from dotenv import load_dotenv
from pypdf import PdfReader
import chromadb
from huggingface_hub import InferenceClient

load_dotenv()

# ── Your imports below ──


True

---
<a id="2-loading"></a>
## 2. Load Job Descriptions and Resume

**Required:**
- 10+ JD files in `data/job_descriptions/` (each as a separate .txt or .pdf)
- Your resume in `data/resume/`
- A metadata file `data/jd_metadata.csv` with columns: filename, company, title, source_url, date_collected

Print: number of JDs loaded, number of resume docs, and preview content from each.

In [3]:
# ── Load JD metadata ──


In [4]:
# ── Load JD documents and resume ──


In [5]:
# ── Preview sample content ──


---
<a id="3-chunking"></a>
## 3. Text Chunking

Split your documents into chunks.  
**Required:** Try at least 2 chunking strategies, compare them quantitatively, and justify your final choice.

**Hint:** JDs often have natural sections (Requirements, Responsibilities, Qualifications). Consider whether your splitter respects these boundaries.

In [6]:
# ── Strategy 1 ──


In [7]:
# ── Strategy 2 ──


In [8]:
# ── Compare strategies ──


### Chunking Decision

**Which strategy did you choose?**  
**Why?**  
**Final settings (chunk_size, overlap):**

---
<a id="4-embedding"></a>
## 4. Embedding and Vector Store

Embed your chunks and store them in a vector database (ChromaDB or FAISS).

**Paid path:** OpenAI `text-embedding-3-small`  
**Free path:** `sentence-transformers/all-MiniLM-L6-v2`

After creating the store, run a test similarity search to verify it works.

In [9]:
#llama 
#huggingface 
#OpenAI
#JUST PAY THE $5 

In [10]:
# ── Create embeddings and vector store ──


In [11]:
# ── Verify: run a test similarity search ──


---
<a id="5-analysis"></a>
## 5. Analysis Prompts and Chain

Build 3 analysis types, each with its own prompt:

1. **Skill Gap Report:** Compare resume skills vs. JD requirements. Output matching skills, missing skills, and recommended actions.
2. **Keyword Alignment:** Extract key terms from a JD, check which appear in the resume, report a match rate.
3. **Fit Summary:** 3-4 sentence narrative assessment citing evidence from both documents.

You also need to wire up the LLM and a way to pass a specific JD + resume into each prompt.

**Required:** Document at least 3 prompt iterations total (across any analysis type) with rationale.

**Reminder:** Prompt design must be your own work (Tier 2 — AI prohibited for this step).

In [12]:
# ── Initialize LLM ──


In [13]:
# ── Analysis 1: Skill Gap Report ──


In [14]:
# ── Analysis 2: Keyword Alignment ──


In [15]:
# ── Analysis 3: Fit Summary ──


### Prompt Iteration Log

Document at least 3 total iterations across any of the analysis types.

**Iteration 1:** [Which analysis? What changed? Why? What improved?]

**Iteration 2:** [Which analysis? What changed? Why? What improved?]

**Iteration 3:** [Which analysis? What changed? Why? What improved?]

---
<a id="6-comparison"></a>
## 6. Zero-shot vs. Few-shot Comparison

Pick one of your 3 analysis types. Create a few-shot version by adding 1-2 example input/output pairs to the prompt. Run both versions on the same JD and compare outputs.

**Reminder:** You must write the few-shot examples yourself (Tier 2).

In [16]:
# ── Few-shot version of your chosen analysis ──


In [17]:
# ── Run both on the same JD, display side by side ──


### Zero-shot vs. Few-shot Analysis

**Which analysis type did you compare?**

**Which performed better?**

**Why? (use specific examples from the outputs above)**

---
<a id="7-evaluation"></a>
## 7. Evaluation

Run all 3 analysis types on your **top 3 target JDs** (9 total analyses).

For each, score:
- **Retrieval relevance:** Did it pull the right JD sections? (Yes/Partial/No)
- **Skill identification accuracy:** Are identified skills/gaps correct? (count correct vs. incorrect)
- **Actionability:** Are recommendations specific and useful? (1-5)
- **Faithfulness:** Does output stick to document content? (Faithful/Partial/Hallucinated)

**Reminder:** Evaluation must be your own work (Tier 2 — AI prohibited).

In [18]:
# ── Run 9 analyses (3 JDs x 3 analysis types) ──


In [19]:
# ── Summarize evaluation results ──


### Evaluation Analysis

**Which analysis type worked best?**

**Which JDs produced the best/worst results? Why?**

**Where did the system hallucinate or produce inaccurate results?**

**What would you improve?**

---

## Next Steps

1. Build your Streamlit app (`streamlit_app.py`) using the pipeline from this notebook
2. Write your Technical Manager Memo (`memo.md`)
3. Complete your AI Usage Log (`ai_log.md`)
4. Verify GitHub repository structure and commit count

---
*BSAN 6200 | Spring 2026 | Assignment 5 — Option B*

## Sample Prompt Iteration (No Real JD Files Yet)
Use this section to practice prompt design with 1-2 sample job texts before loading the full JD set.

In [20]:
sample_resume = '''
Data analyst with 2 years of experience in Python, SQL, Tableau, and dashboard development.
Built KPI reporting pipelines and worked with cross-functional business stakeholders.
'''

sample_jd_1 = '''
Business Analyst role: requires SQL, Excel, dashboarding, stakeholder communication, and process documentation.
Preferred: Python and A/B testing exposure.
'''

sample_jd_2 = '''
Data Analyst role: requires Python, SQL, ETL workflow knowledge, Tableau/Power BI, and experimentation support.
Preferred: cloud data warehouse experience.
'''

print('Sample inputs loaded')

Sample inputs loaded


In [21]:
prompt_v1 = f'''
Compare the resume and job description.
Give a fit score out of 100, 3 strengths, and 3 gaps.

Resume:
{sample_resume}

Job Description:
{sample_jd_1}
'''

prompt_v2 = f'''
You are a job-fit evaluator. Use only evidence from the inputs.
Output JSON keys: fit_score, strengths, gaps, improvement_actions.
Each strength and gap must include a short evidence quote.
Do not invent experience not present in the resume.

Resume:
{sample_resume}

Job Description:
{sample_jd_1}
'''

print('Prompt v1:\n', prompt_v1[:400], '...')
print('\nPrompt v2:\n', prompt_v2[:500], '...')

Prompt v1:
 
Compare the resume and job description.
Give a fit score out of 100, 3 strengths, and 3 gaps.

Resume:

Data analyst with 2 years of experience in Python, SQL, Tableau, and dashboard development.
Built KPI reporting pipelines and worked with cross-functional business stakeholders.


Job Description:

Business Analyst role: requires SQL, Excel, dashboarding, stakeholder communication, and process  ...

Prompt v2:
 
You are a job-fit evaluator. Use only evidence from the inputs.
Output JSON keys: fit_score, strengths, gaps, improvement_actions.
Each strength and gap must include a short evidence quote.
Do not invent experience not present in the resume.

Resume:

Data analyst with 2 years of experience in Python, SQL, Tableau, and dashboard development.
Built KPI reporting pipelines and worked with cross-functional business stakeholders.


Job Description:

Business Analyst role: requires SQL, Excel, dashb ...


### Iteration Notes Template
- Version tested:
- What improved:
- What still failed:
- Next prompt change: